In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from imutils.perspective import four_point_transform

def resizer(image,width=500):
    # get width and height
    h,w,c=image.shape
    height=int((h/w)*width)
    size=(width,height)
    image=cv2.resize(image,size)
    return image,size


def document_scanner(image):
    
    img_re,size=resizer(img)

    detail=cv2.detailEnhance(img_re,sigma_s=20,sigma_r=0.15)
    gray=cv2.cvtColor(detail,cv2.COLOR_BGR2GRAY)
    blur=cv2.GaussianBlur(gray,(5,5),0)

    #detecting the edges
    edge_image=cv2.Canny(blur,75,200)

    #morphological transform
    kernel=np.ones((5,5),np.uint8)
    dilate=cv2.dilate(edge_image,kernel,iterations=1)
    closing=cv2.morphologyEx(dilate,cv2.MORPH_CLOSE,kernel)

    #find the contours
    contours, hire=cv2.findContours(closing,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)
    contours= sorted(contours,key=cv2.contourArea,reverse=True)
    for contour in contours:
        peri=cv2.arcLength(contour,True)
        approx=cv2.approxPolyDP(contour,0.02*peri,True)

        if len(approx) ==4:
            four_points= np.squeeze(approx)
            break

    cv2.drawContours(img_re,[four_points],-1,(0,255,0),3)

    #find four points for original image
    multiplier=img.shape[1]/size[0]
    four_points_original=four_points*multiplier
    four_points_original=four_points_original.astype(int)

    wrap_image=four_point_transform(img,four_points_original)

    return wrap_image, four_points_original, img_re, closing

img=cv2.imread('Extracted/frame_0003.jpg')
crop,points, cnt_img, edgeimg=document_scanner(img)
print(points)
cv2.imshow('original image',img)
cv2.imshow('edges image',cnt_img)
cv2.imshow('cropped image',crop)
cv2.waitKey(0)
cv2.destroyAllWindows()

gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
threshold=cv2.adaptiveThreshold(gray,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,31,20)
cv2.imshow('thresholded',threshold)
cv2.waitKey(0)
cv2.destroyAllWindows()

[[295  58]
 [161  61]
 [160  68]
 [293  65]]


In [3]:
import cv2
import numpy as np

def scan_detection(image):
    HEIGHT,WIDTH,c=image.shape
    document_contour=np.array([ [0,0],[WIDTH,0],[WIDTH,HEIGHT],[0,HEIGHT] ])
    gray=cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
    blur=cv2.GaussianBlur(gray,(5,5),0)
    _,threshold=cv2.threshold(blur,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    contours,_ =cv2.findContours(threshold,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)
    contours=sorted(contours,key=cv2.contourArea,reverse=True)

    max_area=0
    for contour in contours:
        area=cv2.contourArea(contour)
        if area > 1000:
            peri=cv2.arcLength(contour,True)
            approx=cv2.approxPolyDP(contour,0.015*peri,True)
            if area > max_area and len(approx)==4:
                document_contour=approx
                max_area=area

    cv2.drawContours(image,[document_contour],-1,(0,255,0),3)

img=cv2.imread('Extracted/frame_0001.jpg')
cv2.imshow('og',img)
cv2.waitKey(0)
cv2.destroyAllWindows()
scan_detection(img)
cv2.imshow('drawn image',img)
cv2.waitKey(0)
cv2.destroyAllWindows()


## For Optical Flow testing

In [2]:
import cv2
import numpy as np
import time

def draw_flow(img, flow, step=16):
    h, w = img.shape[:2]
    y, x = np.mgrid[step/2:h:step, step/2:w:step].reshape(2,-1).astype(int)
    fx, fy= flow[y,x].T

    lines = np.vstack([x, y, x-fx, y-fy]).T.reshape(-1,2,2)
    lines = np.int32(lines + 0.5)

    img_bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.polylines(img_bgr, lines, 0, (0,255,0) )

    for(x1, y1), (x2, y2) in lines:
        cv2.circle(img_bgr, (x1, y1), 1, (0, 255, 0), -1)

    return img_bgr

def draw_hsv(flow):
    h, w = flow.shape[:2]
    fx, fy = flow[:,:,0], flow [:,:,1]

    ang = np.arctan2(fy, fx) + np.pi
    v = np.sqrt(fx*fx +fy*fy)

    hsv = np.zeros( (h, w, 3), np.uint8)
    hsv[...,0] = ang*(180/np.pi/2)
    hsv[...,1] = 255
    hsv[...,2] = np.minimum(v*4, 255)
    bgr =cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    return bgr

cap = cv2.VideoCapture(0)

suc, prev = cap.read()
prevgray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)

while True:
    suc, img = cap.read()
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # start time to calculate FPS
    start = time.time()

    flow = cv2.calcOpticalFlowFarneback(prevgray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    prevgray = gray
    
    # End time
    end = time.time()
    # Calculate the FPS for current frame detection
    fps = 1/(end - start)

    print(f"{fps:.2f} FPS")

    cv2.imshow('flow', cv2.flip(draw_flow(gray, flow), 1) )
    cv2.imshow('flow HSV', cv2.flip(draw_hsv(flow), 1) )
    
    key = cv2.waitKey(5)
    if key == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()

15.41 FPS
5.96 FPS
10.42 FPS
15.02 FPS
13.40 FPS
17.10 FPS
7.53 FPS
7.12 FPS
8.32 FPS
7.03 FPS
7.56 FPS
4.52 FPS
4.70 FPS
3.00 FPS
5.17 FPS
4.46 FPS
6.64 FPS
6.84 FPS
6.45 FPS
6.35 FPS
7.25 FPS
6.53 FPS
6.67 FPS
6.62 FPS
7.85 FPS
7.49 FPS
6.17 FPS
6.49 FPS
6.59 FPS
6.54 FPS
4.34 FPS
4.02 FPS
2.80 FPS
4.85 FPS
5.34 FPS
5.48 FPS
5.62 FPS
4.63 FPS
4.70 FPS
4.71 FPS
5.10 FPS
5.07 FPS
5.35 FPS
5.57 FPS
5.77 FPS
3.62 FPS
5.13 FPS
5.75 FPS
7.04 FPS
14.66 FPS
14.51 FPS
15.97 FPS
14.60 FPS
14.66 FPS
14.82 FPS
14.85 FPS
12.52 FPS
14.20 FPS
15.16 FPS
14.79 FPS
9.93 FPS
6.98 FPS
13.72 FPS
15.05 FPS
15.54 FPS
14.71 FPS
17.43 FPS
16.21 FPS
16.87 FPS
17.41 FPS
17.53 FPS
21.10 FPS
16.90 FPS
15.83 FPS
17.71 FPS
18.52 FPS
19.84 FPS
17.57 FPS
15.72 FPS
17.39 FPS
17.69 FPS
14.64 FPS
20.11 FPS
18.12 FPS
17.49 FPS
17.83 FPS
18.40 FPS
17.68 FPS
19.37 FPS
17.96 FPS
17.07 FPS
17.68 FPS
19.29 FPS
18.51 FPS
17.92 FPS
17.63 FPS
17.93 FPS
18.77 FPS
17.66 FPS
16.53 FPS
20.61 FPS
19.44 FPS
19.19 FPS
19.01 FPS
19.61 

In [1]:
import cv2
import os
import numpy as np

# Setup
video_path = "ProjectVideo2.mp4"
output_folder = "Extracted"
os.makedirs(output_folder, exist_ok=True)

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
STABLE_LIMIT = 0.15 # Adjust based on noise
page_count = 0

def get_gray_at_time(seconds):
    """Utility to grab a grayscale frame at a specific timestamp."""
    cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)
    ret, frame = cap.read()
    if not ret: return None, None
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Downscale for speed
    gray_small = cv2.resize(gray, (320, 180))
    return gray_small, frame

def calculate_motion(gray1, gray2):
    """Calculates optical flow between two frames."""
    if gray1 is None or gray2 is None: return 999.0
    flow = cv2.calcOpticalFlowFarneback(gray1, gray2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    return np.mean(np.abs(flow))

def find_stable_point(temp_sec):
    """
    Recursively checks temp-0.1 and temp+0.1 to find 
    the 'center' of a stable window.
    """
    g_mid, f_mid = get_gray_at_time(temp_sec)
    g_prev, _ = get_gray_at_time(temp_sec - 0.1)
    g_next, _ = get_gray_at_time(temp_sec + 0.1)

    motion_back = calculate_motion(g_prev, g_mid)
    motion_fwd = calculate_motion(g_mid, g_next)

    # Logic: If both sides are stable, we are in the heart of the page
    if motion_back < STABLE_LIMIT and motion_fwd < STABLE_LIMIT:
        return temp_sec, f_mid
    
    # If the back is unstable, the page just arrived; move forward
    if motion_back >= STABLE_LIMIT:
        return find_stable_point(temp_sec + 0.1)
    
    # If the front is unstable, the page is about to turn; move backward
    if motion_fwd >= STABLE_LIMIT:
        return find_stable_point(temp_sec - 0.1)
    
    return None, None

# --- MAIN EXECUTION ---
current_time = 0.1
while True:
    # Phase 1: Linear search for the first/next stable image
    g0, _ = get_gray_at_time(current_time - 0.1)
    g1, f1 = get_gray_at_time(current_time)
    
    if g1 is None: break # End of video

    if calculate_motion(g0, g1) < STABLE_LIMIT:
        # STABLE IMAGE FOUND
        page_count += 1
        cv2.imwrite(f"{output_folder}/page_{page_count}.jpg", f1)
        print(f"Page {page_count} saved at {current_time:.2f}s")

        # Phase 2: The 2-second Jump & Recursive Adjustment
        jump_time = current_time + 2.0
        stable_time, stable_frame = find_stable_point(jump_time)
        
        if stable_time:
            current_time = stable_time
            # Note: The loop will continue and save this stable_frame in the next iteration
        else:
            current_time += 0.1 # Fallback if jump lands in void
    else:
        current_time += 0.1

cap.release()

Page 1 saved at 10.20s


KeyboardInterrupt: 